# 🦠 COVID-19 Trend Analysis & ARIMA Forecasting
**Author:** Vanil B. Patel | KSV University

Time-series analysis and 30-day forecasting on global COVID-19 data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
print('✅ Libraries loaded!')

In [ ]:
# Generate synthetic COVID-19 time series data
np.random.seed(42)
countries = ['India', 'USA', 'Brazil', 'UK', 'Germany', 'France', 'Italy', 'Russia', 'Japan', 'Australia']
dates = pd.date_range('2020-03-01', '2023-12-31', freq='D')

def gen_wave_series(n, seed):
    np.random.seed(seed)
    t = np.linspace(0, 12, n)
    # 3 waves with noise
    wave = (np.sin(t * 0.8) + np.sin(t * 1.6 + 1) + np.sin(t * 2.4 + 2)) * 10000
    noise = np.random.normal(0, 2000, n)
    series = np.abs(wave + noise + 15000)
    return series.astype(int)

data = {}
for i, country in enumerate(countries):
    multiplier = np.random.uniform(0.2, 5.0)
    data[country] = (gen_wave_series(len(dates), i) * multiplier).astype(int)

df = pd.DataFrame(data, index=dates)
df.index.name = 'date'

print(f'Dataset: {df.shape} ({df.shape[0]} days × {df.shape[1]} countries)')
df.head()

In [ ]:
# ── Daily Cases Trend ──
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

colors = sns.color_palette('tab10', 10)
for i, country in enumerate(countries):
    weekly = df[country].resample('W').mean()
    axes[i].plot(weekly.index, weekly.values, color=colors[i], linewidth=1.5)
    axes[i].fill_between(weekly.index, weekly.values, alpha=0.2, color=colors[i])
    axes[i].set_title(country, fontweight='bold', fontsize=11)
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)

plt.suptitle('Weekly COVID-19 Cases — 10 Countries (2020–2023)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('covid_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ARIMA Forecasting for India ──
country = 'India'
series = df[country].resample('W').mean()

# ADF test for stationarity
adf_result = adfuller(series.dropna())
print(f'ADF Statistic: {adf_result[0]:.4f}')
print(f'p-value: {adf_result[1]:.4f}')
print(f'Stationary: {adf_result[1] < 0.05}')

# Fit ARIMA model
train = series[:-8]  # last 8 weeks for test
test = series[-8:]

model = ARIMA(train, order=(2, 1, 2))
result = model.fit()

# Forecast 30 days (4 weeks + 30-day horizon)
forecast = result.forecast(steps=12)
conf_int = result.get_forecast(steps=12).conf_int()

# Calculate MAPE on test set
test_forecast = result.forecast(steps=len(test))
mape = np.mean(np.abs((test.values - test_forecast) / test.values)) * 100
print(f'\n📊 MAPE on test set: {mape:.2f}%')

In [ ]:
# ── Plot Forecast ──
plt.figure(figsize=(12, 5))

# Historical
plt.plot(train[-52:].index, train[-52:].values, label='Historical', color='steelblue', linewidth=2)
plt.plot(test.index, test.values, label='Actual (test)', color='green', linewidth=2, linestyle='--')

# Forecast
forecast_dates = pd.date_range(series.index[-1], periods=13, freq='W')[1:]
plt.plot(forecast_dates, forecast, label='ARIMA Forecast', color='red', linewidth=2)
plt.fill_between(forecast_dates,
                 conf_int.iloc[:, 0],
                 conf_int.iloc[:, 1],
                 alpha=0.2, color='red', label='95% CI')

plt.title(f'{country} — COVID-19 30-Day Forecast (ARIMA, MAPE={mape:.1f}%)', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Weekly Cases')
plt.legend()
plt.tight_layout()
plt.savefig('arima_forecast.png', dpi=150, bbox_inches='tight')
plt.show()